# Windowed detector completion

The flow matcher described in [Basic concepts](../../user_guide/basic_concepts.rst) is local by construction. It propagates stabilizers fragment by fragment and closes a detector when a propagated flow commutes with the collapsing operations it meets at a fragment boundary. Some valid detectors are invisible to this procedure. This notebook demonstrates the failure on the smallest circuit that exhibits it, then explains the completion pass in `tqecd.window` that recovers the missing detectors.

## Detectors that only exist as products of flows

A detector is a set of measurements whose parity is deterministic in the absence of errors. In flow language it is a flow with trivial input and trivial output. Nothing enters, nothing leaves, and the sign information is carried entirely by a set of measurement records.

The matcher assembles detectors from flows that commute with every collapsing operation they encounter. A flow that anticommutes with a collapse cannot be closed at that boundary and is discarded. Flows, however, form a group under multiplication, and commutation is not preserved by products. If two flows each anticommute with the same collapsing operation, their product commutes with it. A detector may therefore exist only as a product of flows that are each individually rejected, and pairwise flow matching never sees it.

The Y basis transition round is the canonical example. At the folded corner of the patch a single data qubit is measured in the Y basis with an `MY` instruction. The X type stabilizer flow arrives at the corner as $X$ and anticommutes with the measurement. The Z type flow arrives as $Z$ and also anticommutes. Their product arrives as $Y$ up to phase, commutes, and closes into a deterministic parity.


## A minimal example, Y basis memory at distance 3

The smallest circuit exhibiting the mechanism is a single rotated surface code patch that stores a logical qubit between a Y basis initialization and a Y basis measurement (fixed bulk convention, distance 3). It has exactly one folded corner and one collapse per transition, with no spatial arms and no cross corner combinations. The circuit ships with the package as a regression fixture.


In [ ]:
from pathlib import Path

import stim

from tqecd.fragment import split_stim_circuit_into_fragments
from tqecd.utils import remove_annotations

circuit = stim.Circuit(
    Path(
        "../../../src/tqecd/test_files/valid/window/"
        "ymem_y_init_y_meas_k1_fixed_bulk.stim"
    ).read_text()
)
bare = remove_annotations(
    circuit, annotations_to_remove=frozenset({"DETECTOR", "OBSERVABLE_INCLUDE"})
)
fragments = split_stim_circuit_into_fragments(bare)
print(circuit.num_qubits, "qubits,", circuit.num_measurements, "measurements")
print("measurements per fragment", [f.num_measurements for f in fragments])

The circuit splits into six fragments and the third and fourth are the two transition rounds. The initialization round applies `RY` to the corner qubit at coordinate $(1, 5)$ together with controlled Y couplings along the fold, and the measurement round mirrors it with `MY` on the same qubit. You can step through the circuit below with the left and right arrow keys, where each `TICK` is one moment.


In [ ]:
from IPython.display import IFrame

IFrame(circuit.to_crumble_url(), width="100%", height=650)

To see the mechanism, select the corner qubit a few moments before the `MY` and drop a Pauli marker on it, then step across the measurement.

* An $X$ marker anticommutes with `MY`, so the flow stays open and no detector forms.
* A $Z$ marker anticommutes with `MY`, so the flow stays open and no detector forms.
* A $Y$ marker commutes with `MY` and closes into the transition parity.

Running the annotation with and without the completion pass makes the consequence concrete. Passing `window=1` reproduces the historical behaviour of plain flow matching, and `window=2` is the production default.


In [ ]:
from tqecd.construction import annotate_detectors_automatically

historical = annotate_detectors_automatically(bare, window=1)
completed = annotate_detectors_automatically(bare, window=2)
print("window=1 (flow matching only) finds", historical.num_detectors, "detectors")
print("window=2 (the default) finds", completed.num_detectors, "detectors")

The two additional detectors are exactly the transition parities, one per transition round.


In [ ]:
def detector_record_sets(annotated):
    """Absolute measurement record set of every DETECTOR instruction."""
    sets, running = [], 0
    for inst in annotated.flattened():
        if inst.name == "DETECTOR":
            sets.append(
                frozenset(
                    running + t.value
                    for t in inst.targets_copy()
                    if t.is_measurement_record_target
                )
            )
        elif stim.gate_data(inst.name).produces_measurements:
            running += sum(1 for t in inst.targets_copy() if not t.is_combiner)
    return sets


measured_qubit = [
    t.qubit_value
    for inst in bare.flattened()
    if stim.gate_data(inst.name).produces_measurements
    for t in inst.targets_copy()
    if not t.is_combiner
]
coords = circuit.get_final_qubit_coordinates()
added = sorted(
    set(detector_record_sets(completed)) - set(detector_record_sets(historical)),
    key=min,
)
for records in added:
    qubit_coords = [tuple(coords[measured_qubit[r]]) for r in sorted(records)]
    print("records", sorted(records), "measuring qubits at", qubit_coords)

Records 16 to 25 belong to the first transition round and records 26 to 36 to the second, so each added detector straddles the boundary between the two transition rounds and no single fragment boundary sees a commuting closure. Their absence is not benign. Under uniform depolarizing noise, and with the logical parity of the fixture reattached as an observable, the shortest undetected logical error shrinks dramatically.


In [ ]:
def with_uniform_depolarization(c, p=0.001):
    noisy = stim.Circuit()
    qubits = list(range(c.num_qubits))
    for inst in c.flattened():
        if inst.name == "TICK":
            noisy.append("DEPOLARIZE1", qubits, p)
        noisy.append(inst)
    return noisy


# The fixture stores the logical parity as OBSERVABLE_INCLUDE annotations. They were
# stripped together with the detectors, so reattach them before asking for distances.
observable_records, running = [], 0
for inst in circuit.flattened():
    if inst.name == "OBSERVABLE_INCLUDE":
        observable_records.extend(
            running + t.value
            for t in inst.targets_copy()
            if t.is_measurement_record_target
        )
    elif stim.gate_data(inst.name).produces_measurements:
        running += sum(1 for t in inst.targets_copy() if not t.is_combiner)

for name, annotated in (("window=1", historical), ("window=2", completed)):
    with_observable = annotated.copy()
    with_observable.append(
        "OBSERVABLE_INCLUDE",
        [stim.target_rec(r - running) for r in sorted(observable_records)],
        0,
    )
    noisy = with_uniform_depolarization(with_observable)
    print(name, "shortest graphlike error weight", len(noisy.shortest_graphlike_error()))

With flow matching alone a single $Z$ error on a data qubit along the fold, for instance at $(3, 3)$ during the transition rounds, flips the logical observable without triggering any detector. The patch nominally has distance 3, yet the annotated circuit has distance 1. The completion pass restores the full distance.


## Completion pass details

The pass runs by default inside `annotate_detectors_automatically`, after flow matching, and it only ever adds detectors. Candidate generation is delegated to Stim. For every pair of consecutive fragments the sub circuit is rebuilt and `stim.Circuit.flow_generators` is asked for its flow generators. A generator whose input and output Pauli strings are both trivial is a measurement parity that is deterministic no matter which state entered the window, so it is a valid detector of the full circuit. Stim cannot return a nondeterministic parity, which makes every candidate sound. Here is the window covering the two transition rounds.


In [ ]:
import itertools

start = sum(f.num_measurements for f in fragments[:2])
window_circuit = fragments[2].circuit + fragments[3].circuit
candidates = [
    frozenset(start + m for m in flow.measurements_copy())
    for flow in window_circuit.flow_generators()
    if not flow.input_copy().weight and not flow.output_copy().weight
]
for candidate in sorted(candidates, key=min):
    print("candidate", sorted(candidate))

narrow, wide = added[1], added[0]
print()
print("missing detector", sorted(narrow), "is itself a candidate,", narrow in candidates)
print("missing detector", sorted(wide), "is itself a candidate,", wide in candidates)
pair = next(
    (c, d) for c, d in itertools.combinations(candidates, 2) if (c ^ d) == wide
)
print(
    "but it is the product of the candidates",
    sorted(pair[0]),
    "and",
    sorted(pair[1]),
)

These nine parities are all sound detectors, but they are redundant and some of them stretch across the patch. The pruning step works on the record sets as bit vectors over $\mathbb{F}_2$, where the product of two parities is the XOR of their vectors and any sequence of XORs changes only the representatives, never the space they generate. Overlapping candidates are greedily XORed together whenever the combination shrinks a candidate's physical bounding box, which is exactly the operation that recovers the wider transition parity above from two patch crossing generators. The localized survivors are then considered smallest first, any candidate physically wider than the widest detector the flow matcher already found is rejected, and the rest are inserted into an incremental Gaussian elimination seeded with the matched detectors, so only independent new parities are emitted. On this circuit the procedure keeps the 48 matched detectors and adds exactly the two transition parities.

The width based rejection matters because determinism alone cannot separate a detector from a logical observable. The logical parity of this memory experiment is itself a deterministic measurement parity, and a whole circuit flow computation would happily return it as one more generator. Small windows and the width cap are what keep patch wide parities out of the detector set.

A few practical notes to close. Circuits containing `REPEAT` blocks are annotated loop aware first and unrolled only when the looped result is missing detectors or loop body matching fails. The width cap is a heuristic, so a circuit whose only missing detector is physically larger than everything the flow matcher found would have it rejected, and gadget level regression tests remain the safety net. The smallest first ordering with incremental insertion is adapted from the [stim-floquet](https://github.com/jerrylvx/stim-floquet) package of B. Lu (MIT license).
